# Qwen3-4B KV cache ablation — Colab A100

Select **Runtime → Change runtime type → A100 GPU**. This notebook installs an isolated Python 3.11 environment, validates correctness, then benchmarks both modes. No benchmark numbers are prefilled.

Default: input lengths 64–32,768, 128 generated tokens, batch 1, BF16, seed 42, CUDA graphs on in both modes. The uncached graph uses causal padding; see the repository's detailed benchmark README. Long-context uncached runs can take substantial time. Keep the GPU idle and save your ZIP before the Colab runtime expires.


In [ ]:
import os, sys, subprocess
from pathlib import Path

def run(args, **kwargs):
    subprocess.run([str(x) for x in args], check=True, **kwargs)

run(["nvidia-smi"])
repo = Path("/content/tiny-vllm")
if not repo.exists():
    run(["git", "clone", "--branch", "feat/qwen3-kv-cache-ablation",
         "https://github.com/SnorlaxPapa/tiny-vllm.git", repo])
os.chdir(repo)
run(["git", "status", "--short", "--branch"])
run(["git", "rev-parse", "HEAD"])


## Install dependencies

This avoids modifying Colab's Torch stack. FlashAttention may compile from source. A100 and a compatible CUDA toolkit/driver are required; stop and inspect any install error before continuing.


In [ ]:
run([sys.executable, "-m", "pip", "install", "uv"])
uv = [sys.executable, "-m", "uv"]
venv = repo / ".venv-benchmark"
if not venv.exists():
    run(uv + ["venv", "--python", "3.11", venv])
bench_python = venv / "bin/python"
run(uv + ["pip", "install", "--python", bench_python, "torch==2.6.0",
          "--index-url", "https://download.pytorch.org/whl/cu124"])
run(uv + ["pip", "install", "--python", bench_python,
          "-r", "requirements-benchmark.txt", "setuptools", "wheel", "psutil"])
run(uv + ["pip", "install", "--python", bench_python,
          "flash-attn==2.7.4.post1", "--no-build-isolation"],
    env=dict(os.environ, MAX_JOBS="2"))
run([bench_python, "-c", "import torch, flash_attn; assert torch.cuda.is_available(); print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name())"])


## Download the model and freeze its revision

The downloader resolves `main` to an immutable Hugging Face commit and writes that SHA into the local run configs. Pass `--revision <recorded_sha>` to reproduce an earlier run. This downloads several GB of weights.


In [ ]:
run([bench_python, "benchmarks/kv_cache/prepare_model.py"])


## Correctness gate

Run CPU tests, then compare identical token histories against Hugging Face with all four cache/graph combinations. A failure stops this cell; do not treat subsequent performance results as validated. CUDA tests include crossing a 256-token cache block boundary.


In [ ]:
run([bench_python, "-m", "pytest", "-q", "tests"])
run([bench_python, "benchmarks/kv_cache/validate_gpu.py", "--model", "checkpoints/Qwen3-4B",
     "--out", "results/validation.json"])


## Smoke test

64/256 input tokens, 8 output tokens, one warmup/repeat. Each output directory must be new; change the name if rerunning. Worker logs are written as the run progresses.


In [ ]:
run([bench_python, "benchmarks/kv_cache/run.py", "--config", "benchmarks/kv_cache/smoke.local.json",
     "--out", "results/smoke"])


## Full A100 experiment

All ten input lengths, cache on/off, two warmups and three measured repeats. This launches one fresh process for each input-length/mode pair. Raw results are flushed after each repeat. Uncached long contexts are expensive; no runtime estimate has been measured.


In [ ]:
run([bench_python, "benchmarks/kv_cache/run.py", "--config", "benchmarks/kv_cache/a100.local.json",
     "--out", "results/kv_cache_a100"])


## Inspect measured results

The results README includes measured tables, plots and coverage gaps. If a run failed, inspect `results/kv_cache_a100/logs` and use the download cell to save partial results. Missing cases are never rendered as zero.


In [ ]:
from IPython.display import display, Image, Markdown
result_dir = repo / "results/kv_cache_a100"
run([bench_python, "benchmarks/kv_cache/plot.py", "--results", result_dir])
if (result_dir / "metrics.png").exists():
    display(Image(filename=str(result_dir / "metrics.png")))
display(Markdown((result_dir / "README.md").read_text()))


## Optional eager sensitivity check

The main no-cache CUDA graph recomputes padded positions up to a fixed capacity. This control turns graphs off in **both** modes and recomputes the exact context length. Set `RUN_EAGER = True` to run it; optionally reduce `prompt_lengths` in a copied config for a smaller control study.


In [ ]:
RUN_EAGER = False
if RUN_EAGER:
    run([bench_python, "benchmarks/kv_cache/run.py", "--config", "benchmarks/kv_cache/eager.local.json",
         "--out", "results/kv_cache_eager"])


## Download all results

Includes CSVs, plots, configuration, environment details, exact prompts/output IDs, validation JSON and logs. Run this cell even after an interrupted or failed benchmark to preserve partial results.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/kv_cache_results", "zip", root_dir=repo / "results")
files.download(archive)
